# Experiment 35 - Digit-Aware Identity Encoding

Targeted extension of the Experiment 23B identity encoding pipeline.

Experiment 33 showed that digit decomposition produced a small improvement from 0.945243 to 0.945331. This experiment tests whether those digit patterns become more useful when represented through target and frequency encoding.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

TRAIN_PATH = '../data/train.csv'
TARGET = 'Will_Buy_EV'
ID_COL = 'id'

train = pd.read_csv(TRAIN_PATH)

target_values = train[TARGET].astype(str).str.strip()
y = target_values.map({'No': 0, 'Yes': 1}).astype(int)

numeric_cols = [
    'Age',
    'Annual_Income_USD',
    'Daily_Commute_km',
    'Number_of_Cars_Owned',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work',
    'Environmental_Concern_Level'
]

categorical_cols = [
    'Gender',
    'City_Type',
    'Current_Car_Type',
    'Home_Charging_Possible',
    'Subsidy_Available',
    'Range_Anxiety_Level'
]

X_raw = train.drop(columns=[TARGET, ID_COL]).copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training rows:', len(X_train))
print('Validation rows:', len(X_valid))

Training rows: 534932
Validation rows: 133733


In [2]:
# ------------------------------------------------------------
# BASE 23B IDENTITY ENCODING
# ------------------------------------------------------------

def identity_key(series):
    return series.astype('string').fillna('__MISSING__')

def fit_target_mapping(series, target, smoothing=20):
    key = identity_key(series)
    tmp = pd.DataFrame({'key': key, 'target': target.values})
    stats = tmp.groupby('key')['target'].agg(['count', 'mean'])
    global_mean = float(target.mean())
    stats['encoded'] = (
        stats['count'] * stats['mean'] + smoothing * global_mean
    ) / (stats['count'] + smoothing)
    return stats['encoded'], global_mean

def apply_target_mapping(series, mapping, global_mean):
    return identity_key(series).map(mapping).fillna(global_mean).astype(float)

def fit_frequency_mapping(series):
    return identity_key(series).value_counts(normalize=True)

def apply_frequency_mapping(series, mapping):
    return identity_key(series).map(mapping).fillna(0.0).astype(float)

def add_oof_identity_features(X_tr, y_tr, X_va, columns, prefix='identity'):
    X_tr = X_tr.copy()
    X_va = X_va.copy()

    skf = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    for col in columns:
        oof_target = np.zeros(len(X_tr), dtype=float)

        for fit_idx, fold_idx in skf.split(X_tr, y_tr):
            mapping, global_mean = fit_target_mapping(
                X_tr.iloc[fit_idx][col],
                y_tr.iloc[fit_idx],
                smoothing=20
            )

            oof_target[fold_idx] = apply_target_mapping(
                X_tr.iloc[fold_idx][col],
                mapping,
                global_mean
            ).values

        X_tr[f'{prefix}_{col}_target'] = oof_target

        full_mapping, full_global = fit_target_mapping(
            X_tr[col], y_tr, smoothing=20
        )
        X_va[f'{prefix}_{col}_target'] = apply_target_mapping(
            X_va[col], full_mapping, full_global
        ).values

        freq = fit_frequency_mapping(X_tr[col])
        X_tr[f'{prefix}_{col}_frequency'] = apply_frequency_mapping(
            X_tr[col], freq
        )
        X_va[f'{prefix}_{col}_frequency'] = apply_frequency_mapping(
            X_va[col], freq
        )

    return X_tr, X_va

In [3]:
# ------------------------------------------------------------
# DIGIT-DERIVED IDENTITY KEYS
# ------------------------------------------------------------

digit_source_cols = [
    'Age',
    'Annual_Income_USD',
    'Daily_Commute_km',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work'
]

def add_digit_identity_keys(df, columns):
    out = df.copy()

    for col in columns:
        values = pd.to_numeric(out[col], errors='coerce')
        safe = values.fillna(0).abs().astype(np.int64)
        strings = safe.astype(str)

        out[f'{col}__last2_key'] = (safe % 100).astype('string')
        out[f'{col}__last3_key'] = (safe % 1000).astype('string')

        out[f'{col}__digit_sum_key'] = strings.apply(
            lambda x: str(sum(int(ch) for ch in x))
        ).astype('string')

        first_digit = strings.str[0]
        last_digit = (safe % 10).astype(str)
        out[f'{col}__first_last_key'] = (
            first_digit + '_' + last_digit
        ).astype('string')

        digit_count = strings.str.len().astype(str)
        out[f'{col}__count_last_key'] = (
            digit_count + '_' + last_digit
        ).astype('string')

        missing = values.isna()
        key_cols = [
            f'{col}__last2_key',
            f'{col}__last3_key',
            f'{col}__digit_sum_key',
            f'{col}__first_last_key',
            f'{col}__count_last_key'
        ]
        for key_col in key_cols:
            out.loc[missing, key_col] = '__MISSING__'

    return out

X_train = add_digit_identity_keys(X_train, digit_source_cols)
X_valid = add_digit_identity_keys(X_valid, digit_source_cols)

digit_identity_cols = []
for col in digit_source_cols:
    digit_identity_cols.extend([
        f'{col}__last2_key',
        f'{col}__last3_key',
        f'{col}__digit_sum_key',
        f'{col}__first_last_key',
        f'{col}__count_last_key'
    ])

print('Digit identity keys:', len(digit_identity_cols))

Digit identity keys: 25


In [4]:
# ------------------------------------------------------------
# BASE IDENTITY FEATURES + DIGIT IDENTITY FEATURES
# ------------------------------------------------------------

X_train, X_valid = add_oof_identity_features(
    X_train,
    y_train,
    X_valid,
    numeric_cols,
    prefix='base'
)

X_train, X_valid = add_oof_identity_features(
    X_train,
    y_train,
    X_valid,
    digit_identity_cols,
    prefix='digit'
)

print('Final feature count:', X_train.shape[1])

Final feature count: 102


In [5]:
# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

base_numeric = numeric_cols

base_identity_features = [
    f'base_{col}_target' for col in numeric_cols
] + [
    f'base_{col}_frequency' for col in numeric_cols
]

digit_identity_features = [
    f'digit_{col}_target' for col in digit_identity_cols
] + [
    f'digit_{col}_frequency' for col in digit_identity_cols
]

numeric_final = base_numeric + base_identity_features + digit_identity_features

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median'))
            ]),
            numeric_final
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))
            ]),
            categorical_cols
        )
    ]
)

Xtr = preprocessor.fit_transform(X_train)
Xva = preprocessor.transform(X_valid)

print('Encoded train shape:', Xtr.shape)
print('Encoded validation shape:', Xva.shape)

model = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.04,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

model.fit(Xtr, y_train)

pred = model.predict_proba(Xva)[:, 1]
score = roc_auc_score(y_valid, pred)

print('\n' + '=' * 60)
print('EXPERIMENT 35 RESULT')
print('=' * 60)
print(f'35A_Digit_Identity_Encoding ROC-AUC: {score:.6f}')
print('23B benchmark: 0.945243')
print('33B benchmark: 0.945331')
print(f'vs 23B: {score - 0.945243:+.6f}')
print(f'vs 33B: {score - 0.945331:+.6f}')

Encoded train shape: (534932, 88)
Encoded validation shape: (133733, 88)

EXPERIMENT 35 RESULT
35A_Digit_Identity_Encoding ROC-AUC: 0.945361
23B benchmark: 0.945243
33B benchmark: 0.945331
vs 23B: +0.000118
vs 33B: +0.000030
